<a href="https://colab.research.google.com/github/whiterabbit077/st554_hw7/blob/main/HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Title:** Homework #7  
**Class:** ST-554 Big Data  
**Date:** 3/31/2026  
**Author:** Anna Giczewska

In this notebook I read in the red and white wine data from the UCI Machine Learning Repository, combine the two datasets, create a wine type variable, and split the data into training and test sets using stratified sampling.

Then I do two modeling tasks: **Regression task**: predict `alcohol` and **Classification task**: predict `wine_type`

For each task I:
- fit several ordinary models
- use cross validation to choose the best ordinary model
- fit regularized models
- compare final models on the test set



**Imports**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import (
    LinearRegression,
    LassoCV,
    RidgeCV,
    ElasticNetCV,
    LogisticRegression
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    log_loss,
    accuracy_score
)

**Read in and combine the data**

In [2]:
# read in the two csv files from UCI
red_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

red = pd.read_csv(red_url, sep=";")
white = pd.read_csv(white_url, sep=";")

# create wine type variable
red["wine_type"] = "red"
white["wine_type"] = "white"

# combine data
wine = pd.concat([red, white], ignore_index=True)

# create numeric version too (helpful later)
wine["is_white"] = (wine["wine_type"] == "white").astype(int)

# basic checks
print(wine.shape)
print(wine["wine_type"].value_counts())
wine.head()

(6497, 14)
wine_type
white    4898
red      1599
Name: count, dtype: int64


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type,is_white
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red,0


I first read in the red and white wine datasets and combined them into one data frame. I also created two versions of the wine type variable: `wine_type` as a text variable (`red` or `white`) and `is_white` as a numeric variable (`1 = white`, `0 = red`).

This will make it easier to use the data for both regression and logistic regression.

**Split the data using stratified sampling**

In [3]:
train_df, test_df = train_test_split(
    wine,
    test_size=0.20,
    random_state=554,
    stratify=wine["wine_type"]
)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining proportions:")
print(train_df["wine_type"].value_counts(normalize=True))

print("\nTest proportions:")
print(test_df["wine_type"].value_counts(normalize=True))

Training shape: (5197, 14)
Test shape: (1300, 14)

Training proportions:
wine_type
white    0.753896
red      0.246104
Name: proportion, dtype: float64

Test proportions:
wine_type
white    0.753846
red      0.246154
Name: proportion, dtype: float64


I used a stratified train/test split so that the proportion of red and white wines is about the same in both sets.

##Regression task: predict alcohol

**Create regression data**

In [4]:
# predictors for regression
reg_base_features = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide",
    "density", "pH", "sulphates", "quality", "is_white"
]

X_train_reg_base = train_df[reg_base_features].copy()
X_test_reg_base = test_df[reg_base_features].copy()

y_train_reg = train_df["alcohol"].copy()
y_test_reg = test_df["alcohol"].copy()

**Make helper functions for interaction and polynomial models**

In [5]:
def make_reg_interactions(df):
    df_new = df.copy()

    # a few interaction terms
    df_new["volatile_x_sulphates"] = df_new["volatile acidity"] * df_new["sulphates"]
    df_new["density_x_sugar"] = df_new["density"] * df_new["residual sugar"]
    df_new["quality_x_type"] = df_new["quality"] * df_new["is_white"]

    return df_new


def make_reg_polynomials(df):
    df_new = df.copy()

    # a few squared terms
    df_new["volatile_sq"] = df_new["volatile acidity"] ** 2
    df_new["sulphates_sq"] = df_new["sulphates"] ** 2
    df_new["residual_sugar_sq"] = df_new["residual sugar"] ** 2
    df_new["quality_sq"] = df_new["quality"] ** 2

    return df_new


def make_reg_both(df):
    df_new = make_reg_interactions(df)
    df_new = make_reg_polynomials(df_new)
    return df_new

To create different multiple linear regression models, I made a few helper functions. One adds interaction terms, one adds polynomial terms, and one adds both. This lets me compare different model forms in a simple way.

**Fit 4 ordinary multiple linear regression models and compare with CV**

In [6]:
# four candidate MLR models
X_train_reg_1 = X_train_reg_base.copy()
X_train_reg_2 = make_reg_interactions(X_train_reg_base)
X_train_reg_3 = make_reg_polynomials(X_train_reg_base)
X_train_reg_4 = make_reg_both(X_train_reg_base)

# stratified folds based on wine type
reg_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=554)
reg_splits = list(reg_cv.split(X_train_reg_base, train_df["wine_type"]))

reg_models = {
    "MLR 1: main effects only": X_train_reg_1,
    "MLR 2: interactions": X_train_reg_2,
    "MLR 3: polynomials": X_train_reg_3,
    "MLR 4: interactions + polynomials": X_train_reg_4
}

reg_cv_results = []

for name, X_now in reg_models.items():
    cv_out = cross_validate(
        LinearRegression(),
        X_now,
        y_train_reg,
        cv=reg_splits,
        scoring=("neg_root_mean_squared_error", "neg_mean_absolute_error")
    )

    reg_cv_results.append({
        "model": name,
        "cv_rmse": -cv_out["test_neg_root_mean_squared_error"].mean(),
        "cv_mae": -cv_out["test_neg_mean_absolute_error"].mean()
    })

reg_cv_results = pd.DataFrame(reg_cv_results).sort_values("cv_rmse")
reg_cv_results

,model,cv_rmse,cv_mae
3,MLR 4: interactions + polynomials,0.455529,0.335415
1,MLR 2: interactions,0.457713,0.336998
2,MLR 3: polynomials,0.460632,0.339584
0,MLR 1: main effects only,0.461910,0.341153


I fit four different multiple linear regression models and used 5-fold cross validation on the training set to compare them. I used RMSE and MAE as my regression metrics. I selected the model with the smallest CV RMSE as my best ordinary multiple linear regression model.

**Pick the best ordinary regression model**

In [7]:
best_reg_name = reg_cv_results.iloc[0, 0]
print("Best ordinary regression model from CV:", best_reg_name)

if best_reg_name == "MLR 1: main effects only":
    X_train_reg_best = X_train_reg_1
    X_test_reg_best = X_test_reg_base.copy()
elif best_reg_name == "MLR 2: interactions":
    X_train_reg_best = X_train_reg_2
    X_test_reg_best = make_reg_interactions(X_test_reg_base)
elif best_reg_name == "MLR 3: polynomials":
    X_train_reg_best = X_train_reg_3
    X_test_reg_best = make_reg_polynomials(X_test_reg_base)
else:
    X_train_reg_best = X_train_reg_4
    X_test_reg_best = make_reg_both(X_test_reg_base)

best_mlr_model = LinearRegression()
best_mlr_model.fit(X_train_reg_best, y_train_reg)

Best ordinary regression model from CV: MLR 4: interactions + polynomials


LinearRegression()

**Fit LASSO, Ridge, and Elastic Net for regression**

In [8]:
# use the same predictors as the best ordinary regression model
alphas = np.logspace(-3, 1, 40)

lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(
        alphas=alphas,
        cv=reg_splits,
        random_state=554,
        max_iter=20000
    ))
])

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(
        alphas=alphas,
        cv=reg_splits,
        scoring="neg_root_mean_squared_error"
    ))
])

elastic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        alphas=alphas,
        l1_ratio=[0.1, 0.5, 0.9],
        cv=reg_splits,
        random_state=554,
        max_iter=20000
    ))
])

lasso_model.fit(X_train_reg_best, y_train_reg)
ridge_model.fit(X_train_reg_best, y_train_reg)
elastic_model.fit(X_train_reg_best, y_train_reg)

print("Best LASSO alpha:", lasso_model.named_steps["model"].alpha_)
print("Best Ridge alpha:", ridge_model.named_steps["model"].alpha_)
print("Best Elastic Net alpha:", elastic_model.named_steps["model"].alpha_)
print("Best Elastic Net l1_ratio:", elastic_model.named_steps["model"].l1_ratio_)

Best LASSO alpha: 0.001
Best Ridge alpha: 0.002030917620904735
Best Elastic Net alpha: 0.001
Best Elastic Net l1_ratio: 0.1


For the regularized regression models, I standardized the predictors first because LASSO, Ridge, and Elastic Net are scale-sensitive methods. I then used cross validation to choose the tuning parameters.

**Compare the 4 final regression models on the test set**

In [9]:
def regression_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae


reg_test_results = []

final_reg_models = {
    "Best ordinary MLR": best_mlr_model,
    "LASSO": lasso_model,
    "Ridge": ridge_model,
    "Elastic Net": elastic_model
}

for name, model in final_reg_models.items():
    preds = model.predict(X_test_reg_best)
    rmse, mae = regression_metrics(y_test_reg, preds)

    reg_test_results.append({
        "model": name,
        "test_rmse": rmse,
        "test_mae": mae
    })

reg_test_results = pd.DataFrame(reg_test_results).sort_values("test_rmse")
reg_test_results

,model,test_rmse,test_mae
0,Best ordinary MLR,0.611825,0.340934
2,Ridge,0.629852,0.341789
1,LASSO,0.661337,0.344757
3,Elastic Net,0.675151,0.344654


I compared my four final regression models on the test set using RMSE and MAE. The model with the lowest RMSE and MAE is the one that predicts alcohol best on the held-out data.

##Classification task: predict wine type

**Create classification data**

In [10]:
clf_base_features = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide",
    "density", "pH", "sulphates", "alcohol", "quality"
]

X_train_clf_base = train_df[clf_base_features].copy()
X_test_clf_base = test_df[clf_base_features].copy()

y_train_clf = train_df["is_white"].copy()
y_test_clf = test_df["is_white"].copy()

**Make helper functions for logistic models**

In [11]:
def make_clf_interactions(df):
    df_new = df.copy()

    df_new["volatile_x_quality"] = df_new["volatile acidity"] * df_new["quality"]
    df_new["chlorides_x_sulphates"] = df_new["chlorides"] * df_new["sulphates"]
    df_new["density_x_fixed"] = df_new["density"] * df_new["fixed acidity"]

    return df_new


def make_clf_polynomials(df):
    df_new = df.copy()

    df_new["residual_sugar_sq"] = df_new["residual sugar"] ** 2
    df_new["chlorides_sq"] = df_new["chlorides"] ** 2
    df_new["density_sq"] = df_new["density"] ** 2
    df_new["alcohol_sq"] = df_new["alcohol"] ** 2

    return df_new


def make_clf_both(df):
    df_new = make_clf_interactions(df)
    df_new = make_clf_polynomials(df_new)
    return df_new

**Fit 4 ordinary logistic models and choose best with CV log-loss**

In [12]:
X_train_clf_1 = X_train_clf_base.copy()
X_train_clf_2 = make_clf_interactions(X_train_clf_base)
X_train_clf_3 = make_clf_polynomials(X_train_clf_base)
X_train_clf_4 = make_clf_both(X_train_clf_base)

clf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=554)
clf_splits = list(clf_cv.split(X_train_clf_base, y_train_clf))

clf_models = {
    "Logistic 1: main effects only": X_train_clf_1,
    "Logistic 2: interactions": X_train_clf_2,
    "Logistic 3: polynomials": X_train_clf_3,
    "Logistic 4: interactions + polynomials": X_train_clf_4
}

clf_cv_results = []

for name, X_now in clf_models.items():
    # using a very large C to act like "almost no penalty"
    log_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=5000,
            random_state=554
        ))
    ])

    cv_out = cross_validate(
        log_model,
        X_now,
        y_train_clf,
        cv=clf_splits,
        scoring=("neg_log_loss", "accuracy")
    )

    clf_cv_results.append({
        "model": name,
        "cv_log_loss": -cv_out["test_neg_log_loss"].mean(),
        "cv_accuracy": cv_out["test_accuracy"].mean()
    })

clf_cv_results = pd.DataFrame(clf_cv_results).sort_values("cv_log_loss")
clf_cv_results

,model,cv_log_loss,cv_accuracy
3,Logistic 4: interactions + polynomials,0.030206,0.995189
1,Logistic 2: interactions,0.031460,0.994804
2,Logistic 3: polynomials,0.031817,0.994420
0,Logistic 1: main effects only,0.031827,0.994804


For the logistic regression models, I used log-loss as the main model selection metric during training, as istructed. I also looked at accuracy as a secondary measure.

**Pick best ordinary logistic model**

In [13]:
best_clf_name = clf_cv_results.iloc[0, 0]
print("Best ordinary logistic model from CV:", best_clf_name)

if best_clf_name == "Logistic 1: main effects only":
    X_train_clf_best = X_train_clf_1
    X_test_clf_best = X_test_clf_base.copy()
elif best_clf_name == "Logistic 2: interactions":
    X_train_clf_best = X_train_clf_2
    X_test_clf_best = make_clf_interactions(X_test_clf_base)
elif best_clf_name == "Logistic 3: polynomials":
    X_train_clf_best = X_train_clf_3
    X_test_clf_best = make_clf_polynomials(X_test_clf_base)
else:
    X_train_clf_best = X_train_clf_4
    X_test_clf_best = make_clf_both(X_test_clf_base)

best_logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=5000,
        random_state=554
    ))
])

best_logistic_model.fit(X_train_clf_best, y_train_clf)

Best ordinary logistic model from CV: Logistic 4: interactions + polynomials


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=1000000.0, max_iter=5000,
                                    random_state=554))])

**Fit regularized logistic models**  

For the penalized logistic models, I’d keep it simple and use 6 predictors. That still satisfies the homework and runs faster.

In [14]:
log_reg_features = [
    "volatile acidity",
    "residual sugar",
    "chlorides",
    "total sulfur dioxide",
    "density",
    "alcohol"
]

X_train_log_pen = train_df[log_reg_features].copy()
X_test_log_pen = test_df[log_reg_features].copy()

**helper function for CV tuning**

In [15]:
def get_log_cv_results(model, X, y, splits):
    cv_out = cross_validate(
        model,
        X,
        y,
        cv=splits,
        scoring=("neg_log_loss", "accuracy")
    )

    return (
        -cv_out["test_neg_log_loss"].mean(),
        cv_out["test_accuracy"].mean()
    )

**LASSO logistic**

In [16]:
l1_choices = [0.01, 0.1, 1, 10]

best_l1_logloss = np.inf
best_l1_C = None

for C_val in l1_choices:
    temp_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=C_val,
            max_iter=5000,
            random_state=554
        ))
    ])

    cv_logloss, cv_acc = get_log_cv_results(temp_model, X_train_log_pen, y_train_clf, clf_splits)

    if cv_logloss < best_l1_logloss:
        best_l1_logloss = cv_logloss
        best_l1_C = C_val

print("Best logistic LASSO C:", best_l1_C)

logistic_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=best_l1_C,
        max_iter=5000,
        random_state=554
    ))
])

logistic_lasso.fit(X_train_log_pen, y_train_clf)

Best logistic LASSO C: 10


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=10, max_iter=5000, penalty='l1',
                                    random_state=554, solver='liblinear'))])

**Ridge logistic**

In [17]:
l2_choices = [0.01, 0.1, 1, 10]

best_l2_logloss = np.inf
best_l2_C = None

for C_val in l2_choices:
    temp_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            C=C_val,
            max_iter=5000,
            random_state=554
        ))
    ])

    cv_logloss, cv_acc = get_log_cv_results(temp_model, X_train_log_pen, y_train_clf, clf_splits)

    if cv_logloss < best_l2_logloss:
        best_l2_logloss = cv_logloss
        best_l2_C = C_val

print("Best logistic Ridge C:", best_l2_C)

logistic_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=best_l2_C,
        max_iter=5000,
        random_state=554
    ))
])

logistic_ridge.fit(X_train_log_pen, y_train_clf)

Best logistic Ridge C: 10


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=10, max_iter=5000, random_state=554))])

**Elastic Net logistic**

In [18]:
elastic_choices = [0.01, 0.1, 1, 10]
l1_ratio_choices = [0.2, 0.5, 0.8]

best_en_logloss = np.inf
best_en_C = None
best_en_l1_ratio = None

for C_val in elastic_choices:
    for l1_val in l1_ratio_choices:
        temp_model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                C=C_val,
                l1_ratio=l1_val,
                max_iter=5000,
                random_state=554
            ))
        ])

        cv_logloss, cv_acc = get_log_cv_results(temp_model, X_train_log_pen, y_train_clf, clf_splits)

        if cv_logloss < best_en_logloss:
            best_en_logloss = cv_logloss
            best_en_C = C_val
            best_en_l1_ratio = l1_val

print("Best logistic Elastic Net C:", best_en_C)
print("Best logistic Elastic Net l1_ratio:", best_en_l1_ratio)

logistic_elastic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=best_en_C,
        l1_ratio=best_en_l1_ratio,
        max_iter=5000,
        random_state=554
    ))
])

logistic_elastic.fit(X_train_log_pen, y_train_clf)

Best logistic Elastic Net C: 10
Best logistic Elastic Net l1_ratio: 0.2


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=10, l1_ratio=0.2, max_iter=5000,
                                    penalty='elasticnet', random_state=554,
                                    solver='saga'))])

**Compare the 4 final classification models on the test set**

In [19]:
clf_test_results = []

# ordinary best logistic uses X_test_clf_best
# regularized logistic models use X_test_log_pen

# best ordinary logistic
best_probs = best_logistic_model.predict_proba(X_test_clf_best)
best_preds = best_logistic_model.predict(X_test_clf_best)

clf_test_results.append({
    "model": "Best ordinary logistic",
    "test_log_loss": log_loss(y_test_clf, best_probs),
    "test_accuracy": accuracy_score(y_test_clf, best_preds)
})

# logistic lasso
lasso_probs = logistic_lasso.predict_proba(X_test_log_pen)
lasso_preds = logistic_lasso.predict(X_test_log_pen)

clf_test_results.append({
    "model": "Logistic LASSO",
    "test_log_loss": log_loss(y_test_clf, lasso_probs),
    "test_accuracy": accuracy_score(y_test_clf, lasso_preds)
})

# logistic ridge
ridge_probs = logistic_ridge.predict_proba(X_test_log_pen)
ridge_preds = logistic_ridge.predict(X_test_log_pen)

clf_test_results.append({
    "model": "Logistic Ridge",
    "test_log_loss": log_loss(y_test_clf, ridge_probs),
    "test_accuracy": accuracy_score(y_test_clf, ridge_preds)
})

# logistic elastic net
elastic_probs = logistic_elastic.predict_proba(X_test_log_pen)
elastic_preds = logistic_elastic.predict(X_test_log_pen)

clf_test_results.append({
    "model": "Logistic Elastic Net",
    "test_log_loss": log_loss(y_test_clf, elastic_probs),
    "test_accuracy": accuracy_score(y_test_clf, elastic_preds)
})

clf_test_results = pd.DataFrame(clf_test_results).sort_values("test_log_loss")
clf_test_results

,model,test_log_loss,test_accuracy
0,Best ordinary logistic,0.041416,0.995385
1,Logistic LASSO,0.056201,0.993077
2,Logistic Ridge,0.056304,0.993846
3,Logistic Elastic Net,0.056309,0.993846


I compared the final classification models on the test set using both log-loss and accuracy. Since the homework specifically asks for log-loss during model selection, I focus mainly on that metric, but I also report accuracy because it is easy to interpret.